# Backtest: UNDER After 3PT Make

**Hypothesis:** When a player hits a 3-pointer and the live line jumps in the next poll (~60s), the UNDER is temporarily mis-priced and our MC model has edge.

**Method:** 
1. Load all signals from S3
2. For each unique game, fetch ESPN PBP (play-by-play with wallclock timestamps)
3. For each UNDER signal, check if the player made a 3PT in the 60s window before the signal timestamp
4. Check if the live line jumped vs the previous poll for that bookmaker
5. Record outcome (final pts vs live_line)
6. Sweep edge thresholds: 5 / 10 / 15 / 20 pp

**Key finding from Steph Curry proof-of-concept:**
- Mechanism confirmed: 3PT makes do cause line jumps (+1 to +5 pts)
- Q1/Q2 makes → 4/4 WIN; Q3/Q4 makes → 0/5 WIN — strong quarter filter signal
- Example: P1 3PT, DK line jumps 21.5→23.5 (+2), Steph scores 17 final → UNDER 23.5 WIN ✓

In [1]:
import subprocess, warnings, time
import duckdb
import pandas as pd
import numpy as np
import requests
import pytz
from datetime import datetime, timezone, timedelta
from collections import defaultdict

warnings.filterwarnings('ignore')
ET = pytz.timezone('US/Eastern')

SESSION = requests.Session()
SESSION.verify = False
SESSION.headers.update({'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'})

# ── parameters ──────────────────────────────────────────────────────────────
WINDOW_SECONDS   = 60    # 3PT must be within this many seconds before the signal
MIN_LINE_JUMP    = 0.5   # live_line must have increased by at least this vs prior poll
MIN_PTS_JUMP     = 3.0   # current_points must have increased by ≥3 vs prior poll (confirms 3PT registered)
MAX_POLL_INTERVAL = 65   # prev→signal poll gap must be ≤ this (60s cadence + 5s latency)
EDGE_THRESHOLDS  = [0.05, 0.10, 0.15, 0.20]   # edge sweep
EXCLUDED_BKS     = {'bovada', 'betonlineag'}   # known stale / no live updates

S3_BUCKET   = 'nba-betting-mt'
S3_SIGNALS  = 'data/04_output/live_betting_signals/player_points'
print('Parameters loaded.')

Parameters loaded.


In [2]:
# ── AWS creds for DuckDB ─────────────────────────────────────────────────────
def _get_cred(key):
    out = subprocess.run(['aws', 'configure', 'get', key], capture_output=True, text=True, timeout=5)
    return out.stdout.strip() if out.returncode == 0 else ''

AK = _get_cred('aws_access_key_id')
SK = _get_cred('aws_secret_access_key')
assert AK and SK, 'AWS credentials not found — run: aws configure'

def duckdb_con():
    con = duckdb.connect(':memory:')
    con.execute('INSTALL httpfs; LOAD httpfs;')
    con.execute("SET s3_region='us-east-2';")
    con.execute(f"SET s3_access_key_id='{AK}';")
    con.execute(f"SET s3_secret_access_key='{SK}';")
    return con

print('AWS creds OK.')

AWS creds OK.


## 1. Load All Signals

In [3]:
con = duckdb_con()
signals = con.execute(f"""
    SELECT *
    FROM read_parquet('s3://{S3_BUCKET}/{S3_SIGNALS}/*.parquet')
    WHERE bet_side = 'UNDER'
      AND bookmaker NOT IN ({', '.join(repr(b) for b in EXCLUDED_BKS)})
      AND (bookmaker_stale IS NULL OR bookmaker_stale = FALSE)
""").fetchdf()
con.close()

signals['ts'] = pd.to_datetime(signals['save_timestamp_utc'], utc=True)
signals = signals.sort_values(['game_id','bookmaker','ts']).reset_index(drop=True)

print(f'Total UNDER signals loaded:   {len(signals):,}')
print(f'Unique games:                  {signals["game_id"].nunique():,}')
print(f'Unique players:                {signals["player_name"].nunique():,}')
print(f'Date range: {signals["game_date_et"].min()} → {signals["game_date_et"].max()}')
print(f'Bookmakers: {sorted(signals["bookmaker"].unique())}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total UNDER signals loaded:   184,191
Unique games:                  423
Unique players:                305
Date range: 2026-02-12 → 2026-05-19
Bookmakers: ['betmgm', 'betrivers', 'draftkings', 'fanatics', 'fanduel', 'williamhill_us']


## 2. Build Per-Signal "Previous Poll" Lookup

For each signal row we need the **previous poll** for the same `(game_id, player_name, bookmaker)` to know:
- Was `current_points` ≥3 higher? (confirms 3PT registered in boxscore)
- Did `live_line` go up? (the line-jump check)

In [4]:
# Load ALL signals (OVER + UNDER) to build the prior-poll lookup
con = duckdb_con()
all_sigs = con.execute(f"""
    SELECT game_id, player_name, bookmaker, save_timestamp_utc, current_points, live_line
    FROM read_parquet('s3://{S3_BUCKET}/{S3_SIGNALS}/*.parquet')
    WHERE bookmaker NOT IN ({', '.join(repr(b) for b in EXCLUDED_BKS)})
""").fetchdf()
con.close()

all_sigs['ts'] = pd.to_datetime(all_sigs['save_timestamp_utc'], utc=True)
all_sigs = all_sigs.sort_values(['game_id','player_name','bookmaker','ts']).reset_index(drop=True)

# Shift within each (game, player, bookmaker) group to get the previous poll's values
grp = all_sigs.groupby(['game_id','player_name','bookmaker'])
all_sigs['prev_current_points'] = grp['current_points'].shift(1)
all_sigs['prev_live_line']      = grp['live_line'].shift(1)
all_sigs['prev_ts']             = grp['ts'].shift(1)   # previous poll timestamp
all_sigs['pts_delta']           = all_sigs['current_points'] - all_sigs['prev_current_points']
all_sigs['line_delta']          = all_sigs['live_line']      - all_sigs['prev_live_line']

# Build a lookup keyed by (game_id, player_name, bookmaker, ts)
prev_lookup = all_sigs.set_index(['game_id','player_name','bookmaker','ts'])[
    ['pts_delta','line_delta','prev_live_line','prev_ts']
]

print(f'Prior-poll lookup built: {len(prev_lookup):,} rows')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Prior-poll lookup built: 424,959 rows


## 3. Fetch ESPN PBP for All Games

One API call per unique game_id. Cached in `pbp_cache` dict so re-running cells is fast.

In [5]:
def fetch_3pt_makes(game_id: str) -> list[dict]:
    """Return list of {player_name_lower, ts_utc, period, clock, text} for all 3PT makes."""
    url = f'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}'
    try:
        resp = SESSION.get(url, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        return []
    plays = resp.json().get('plays', [])
    makes = []
    for p in plays:
        text = p.get('text', '')
        wc   = p.get('wallclock', '')
        if 'makes' not in text.lower():
            continue
        if not ('three point' in text.lower() or '3-pt' in text.lower() or 'three pointer' in text.lower()):
            continue
        if not wc:
            continue
        try:
            ts_utc = datetime.fromisoformat(wc.replace('Z', '+00:00'))
        except Exception:
            continue
        # Extract shooter name: text starts with "<FirstName> <LastName> makes ..."
        name_raw = text.split(' makes ')[0].strip().lower()
        makes.append({
            'player_name_lower': name_raw,
            'ts_utc': ts_utc,
            'period': p.get('period', {}).get('number', 0),
            'clock': p.get('clock', {}).get('displayValue', ''),
            'text': text,
        })
    return makes


def fetch_final_scores(game_id: str) -> dict:
    """Return {normalized_player_name_lower: final_pts} from ESPN box score."""
    url = f'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary?event={game_id}'
    try:
        resp = SESSION.get(url, timeout=15)
        resp.raise_for_status()
    except Exception:
        return {}
    data = resp.json()
    out = {}
    for team in data.get('boxscore', {}).get('players', []):
        for sb in team.get('statistics', []):
            labels = sb.get('labels', [])
            try:
                pts_idx = labels.index('PTS')
            except ValueError:
                continue
            for ath in sb.get('athletes', []):
                name = ath.get('athlete', {}).get('displayName', '').lower()
                stats = ath.get('stats', [])
                if pts_idx < len(stats):
                    try:
                        out[name] = int(stats[pts_idx])
                    except (ValueError, TypeError):
                        pass
    return out


print('ESPN fetch functions defined.')

ESPN fetch functions defined.


In [6]:
game_ids = signals['game_id'].astype(str).unique().tolist()
print(f'Fetching ESPN PBP + box scores for {len(game_ids)} games...')

pbp_cache    = {}   # game_id -> list of 3PT make dicts
scores_cache = {}   # game_id -> {player_name_lower: final_pts}

for i, gid in enumerate(game_ids):
    pbp_cache[gid]    = fetch_3pt_makes(gid)
    scores_cache[gid] = fetch_final_scores(gid)
    if (i + 1) % 20 == 0 or (i + 1) == len(game_ids):
        print(f'  {i+1}/{len(game_ids)} games fetched')
    time.sleep(0.15)   # gentle rate limiting

total_makes = sum(len(v) for v in pbp_cache.values())
print(f'\nTotal 3PT makes across all games: {total_makes:,}')

Fetching ESPN PBP + box scores for 423 games...


  20/423 games fetched


  40/423 games fetched


  60/423 games fetched


  80/423 games fetched


...

  360/423 games fetched


  380/423 games fetched


  400/423 games fetched


  420/423 games fetched


  423/423 games fetched

Total 3PT makes across all games: 9,899


## 4. Tag Signals: Did a 3PT Make Occur in the 60s Window Before?

For each UNDER signal, **three conditions** must all hold:
1. **ESPN 3PT wallclock gate**: a play matching the player's name containing `makes ... three point` has a `wallclock` timestamp within `WINDOW_SECONDS` before the signal timestamp
2. **Line jumped**: `line_delta >= MIN_LINE_JUMP` vs prior poll (market reacted)
3. **Clean poll interval**: `poll_interval_s <= MAX_POLL_INTERVAL` (prev→signal gap ≤ 65s) — ensures the line delta and 3PT detection are from a single 60s polling cycle, not a multi-minute gap where anything could have happened

Note: `pts_delta >= 3` is intentionally **not** a gate — EDA showed 62%+ of such deltas in gaps >90s are multi-play accumulations (1+2, 1+1+1, etc.), not clean 3PT makes.

In [7]:
def normalize_name(name: str) -> str:
    import re
    return re.sub(r"[^a-z ]", "", name.lower()).strip()

def player_names_match(signal_name: str, espn_name: str) -> bool:
    sn = normalize_name(signal_name)
    en = normalize_name(espn_name)
    return sn == en or sn in en or en in sn

tagged_rows = []

for _, row in signals.iterrows():
    gid    = str(row['game_id'])
    pname  = row['player_name']
    bk     = row['bookmaker']
    sig_ts = row['ts']

    # prior-poll deltas
    pts_delta  = float('nan')
    line_delta = float('nan')
    prev_ts    = pd.NaT
    try:
        prior = prev_lookup.loc[(gid, pname, bk, sig_ts)]
        if isinstance(prior, pd.DataFrame):
            prior = prior.iloc[0]
        pts_delta  = float(prior['pts_delta'])
        line_delta = float(prior['line_delta'])
        prev_ts    = prior['prev_ts']
    except KeyError:
        pass

    # Poll interval gate: must be a clean single-cycle poll (≤65s)
    if pd.notna(prev_ts):
        poll_interval_s = (sig_ts - pd.Timestamp(prev_ts)).total_seconds()
    else:
        poll_interval_s = float('nan')

    interval_clean = (not np.isnan(poll_interval_s)) and (poll_interval_s <= MAX_POLL_INTERVAL)
    line_jumped    = (not np.isnan(line_delta)) and (line_delta >= MIN_LINE_JUMP)

    # 3PT make in window — ESPN wallclock is the ONLY reliable 3PT gate
    window_start = sig_ts - timedelta(seconds=WINDOW_SECONDS)
    makes = pbp_cache.get(gid, [])
    trigger_make = None
    for m in makes:
        if window_start <= m['ts_utc'] <= sig_ts:
            if player_names_match(pname, m['player_name_lower']):
                trigger_make = m
                break

    # Trigger: ESPN 3PT confirmed AND line jumped AND clean poll interval
    is_trigger = (trigger_make is not None) and line_jumped and interval_clean

    # final score
    scores = scores_cache.get(gid, {})
    final_pts = None
    for espn_name, pts in scores.items():
        if player_names_match(pname, espn_name):
            final_pts = pts
            break

    outcome = None
    if final_pts is not None:
        outcome = 'WIN' if final_pts < row['live_line'] else 'LOSS'

    tagged_rows.append({
        **row.to_dict(),
        'pts_delta':      pts_delta,
        'line_delta':     line_delta,
        'poll_interval_s': poll_interval_s,
        'prev_ts':        prev_ts,
        'trigger_ts_utc': trigger_make['ts_utc'] if trigger_make else None,
        'interval_clean': interval_clean,
        'line_jumped':    line_jumped,
        'is_trigger':     is_trigger,
        'trigger_period': trigger_make['period'] if trigger_make else None,
        'trigger_clock':  trigger_make['clock']  if trigger_make else None,
        'trigger_text':   trigger_make['text']   if trigger_make else None,
        'final_pts':      final_pts,
        'outcome':        outcome,
    })

tagged   = pd.DataFrame(tagged_rows)
triggers = tagged[tagged['is_trigger']].copy()

print(f'Total UNDER signals:      {len(tagged):,}')
print(f'3PT trigger signals:       {len(triggers):,}')
print(f'  -> with known outcome:    {triggers["outcome"].notna().sum():,}')

# Show how many were dropped by the interval gate (vs just ESPN+line)
no_interval_gate = tagged[(tagged['trigger_ts_utc'].notna()) & tagged['line_jumped']]
print(f'\nWithout interval gate (ESPN+line only): {len(no_interval_gate):,}')
print(f'Dropped by interval gate (>65s):        {len(no_interval_gate) - len(triggers):,}')


Total UNDER signals:      184,191
3PT trigger signals:       410
  -> with known outcome:    410

Without interval gate (ESPN+line only): 810
Dropped by interval gate (>65s):        400


## EDA: Polling Interval Sanity Check

Before trusting `pts_delta`, verify the actual interval between polls. A `pts_delta` of 5+ could mean two scoring plays happened in a single gap (e.g., a 2PT then a 3PT), not just one 3PT. If the poller skipped a minute, the delta is meaningless as a 3PT-specific signal.

In [8]:
# ── Interval distribution across all (game, player, bookmaker) series ──────
grp_intervals = (
    all_sigs
    .sort_values(['game_id','player_name','bookmaker','ts'])
    .groupby(['game_id','player_name','bookmaker'])['ts']
    .diff()
    .dt.total_seconds()
    .dropna()
)

print("=== Poll interval distribution (all game/player/bookmaker series) ===")
print(f"  n intervals:   {len(grp_intervals):,}")
desc = grp_intervals.describe(percentiles=[.25,.5,.75,.90,.95,.99])
for stat, val in desc.items():
    print(f"  {stat:>6}: {val:.1f}s")

# Bucket by interval length
cuts = [0, 50, 70, 90, 120, 180, 300, 600, float('inf')]
labels = ['<50s','50–70s','70–90s','90–120s','2–3min','3–5min','5–10min','>10min']
bucketed = pd.cut(grp_intervals, bins=cuts, labels=labels)
print("\nInterval buckets:")
print(bucketed.value_counts().sort_index().to_string())


=== Poll interval distribution (all game/player/bookmaker series) ===
  n intervals:   404,305


   count: 404305.0s
    mean: 152.7s
     std: 426.8s
     min: 0.0s
     25%: 58.9s
     50%: 60.1s
     75%: 61.8s
     90%: 180.5s
     95%: 595.0s
     99%: 2400.0s
     max: 8176.9s

Interval buckets:
ts
<50s         5553
50–70s     297508
70–90s       7002
90–120s     10136
2–3min      12758
3–5min      11433
5–10min      9874
>10min      19843


In [9]:
# ── Does pts_delta >= 5 correlate with longer intervals? ────────────────────
print("=== pts_delta distribution in trigger signals ===")
print(triggers['pts_delta'].value_counts().sort_index().to_string())

print("\n=== Prior-poll interval for triggers (seconds between polls) ===")
# Compute interval for each trigger signal
trig_intervals = []
for _, row in triggers.iterrows():
    gid   = str(row['game_id'])
    pname = row['player_name']
    bk    = row['bookmaker']
    ts    = row['ts']
    # Find prior ts in all_sigs for same (game, player, bookmaker)
    prior_rows = all_sigs[
        (all_sigs['game_id'] == gid) &
        (all_sigs['player_name'] == pname) &
        (all_sigs['bookmaker'] == bk) &
        (all_sigs['ts'] < ts)
    ]
    if len(prior_rows):
        prior_ts = prior_rows['ts'].max()
        interval = (ts - prior_ts).total_seconds()
    else:
        interval = float('nan')
    trig_intervals.append(interval)

triggers = triggers.copy()
triggers['poll_interval_s'] = trig_intervals

print(triggers['poll_interval_s'].describe(percentiles=[.5,.75,.90,.95,.99]).to_string())

print("\n=== Interval vs pts_delta (mean interval by pts_delta bucket) ===")
triggers['pts_delta_bucket'] = pd.cut(triggers['pts_delta'], bins=[2,3,4,5,6,8,10,float('inf')],
                                       labels=['3','4','5','6','7–8','9–10','>10'])
print(triggers.groupby('pts_delta_bucket', observed=True)['poll_interval_s']
      .agg(['count','mean','median']).to_string())


=== pts_delta distribution in trigger signals ===
pts_delta
0.0    232
1.0     13
2.0     64
3.0     88
4.0      7
5.0      6

=== Prior-poll interval for triggers (seconds between polls) ===


count    410.000000
mean      59.436666
std        2.692050
min       35.132877
50%       59.835774
75%       60.658422
90%       61.485752
95%       62.438384
99%       63.530993
max       64.996710

=== Interval vs pts_delta (mean interval by pts_delta bucket) ===
                  count       mean     median
pts_delta_bucket                             
3                    88  58.994515  59.788453
4                     7  57.554307  58.600851
5                     6  60.507813  60.062782


In [10]:
# ── Flag triggers where interval > 90s (likely multi-play gap) ──────────────
CLEAN_INTERVAL_THRESHOLD = 90   # seconds

triggers['interval_clean'] = triggers['poll_interval_s'] <= CLEAN_INTERVAL_THRESHOLD
print(f"Trigger signals with interval <= {CLEAN_INTERVAL_THRESHOLD}s: {triggers['interval_clean'].sum()} / {len(triggers)}")
print(f"Trigger signals with interval >  {CLEAN_INTERVAL_THRESHOLD}s: {(~triggers['interval_clean']).sum()} (multi-play gap — pts_delta unreliable)")

# Re-evaluate outcomes on CLEAN triggers only
clean = triggers[triggers['interval_clean'] & triggers['outcome'].notna()].copy()
clean = clean.sort_values('ts').drop_duplicates(subset=['game_id','player_name','bookmaker','live_line'], keep='first')
clean['decimal_odds'] = clean['under_odds'].apply(lambda x: (float(x)/100+1) if float(x)>=100 else (100/abs(float(x))+1))
clean['profit'] = clean.apply(lambda r: (r['decimal_odds']-1)*100 if r['outcome']=='WIN' else -100, axis=1)

n_c  = len(clean)
w_c  = (clean['outcome']=='WIN').sum()
roi_c = clean['profit'].sum() / (n_c*100) if n_c else 0

print(f"\n=== CLEAN triggers only (interval <= {CLEAN_INTERVAL_THRESHOLD}s) ===")
print(f"  n:          {n_c}")
print(f"  W–L:        {w_c}–{n_c-w_c}")
print(f"  Hit rate:   {w_c/n_c:.1%}" if n_c else "  Hit rate: n/a")
print(f"  ROI:        {roi_c:+.1%}")

print("\nBy quarter (clean only):")
q_rows = []
for period in sorted(clean['trigger_period'].dropna().unique()):
    sub = clean[clean['trigger_period']==period]
    n_q, w_q = len(sub), (sub['outcome']=='WIN').sum()
    q_rows.append({'period': f'Q{int(period)}', 'n': n_q, 'wins': w_q,
                   'hit_rate': f'{w_q/n_q:.1%}' if n_q else 'n/a',
                   'roi': f'{sub["profit"].sum()/(n_q*100):+.1%}' if n_q else 'n/a'})
print(pd.DataFrame(q_rows).to_string(index=False))

print("\nEdge sweep (clean only):")
for thresh in EDGE_THRESHOLDS:
    sub = clean[clean['edge_after'] >= thresh]
    n_t = len(sub)
    if n_t == 0:
        print(f"  {thresh:.0%}: n=0")
        continue
    w_t = (sub['outcome']=='WIN').sum()
    print(f"  {thresh:.0%}: n={n_t}  W-L={w_t}-{n_t-w_t}  hit={w_t/n_t:.1%}  roi={sub['profit'].sum()/(n_t*100):+.1%}")


Trigger signals with interval <= 90s: 410 / 410
Trigger signals with interval >  90s: 0 (multi-play gap — pts_delta unreliable)

=== CLEAN triggers only (interval <= 90s) ===
  n:          407
  W–L:        169–238
  Hit rate:   41.5%
  ROI:        -22.6%

By quarter (clean only):
period   n  wins hit_rate    roi
    Q1 107    46    43.0% -20.0%
    Q2  60    27    45.0% -15.1%
    Q3 162    70    43.2% -20.0%
    Q4  78    26    33.3% -37.2%

Edge sweep (clean only):
  5%: n=407  W-L=169-238  hit=41.5%  roi=-22.6%
  10%: n=407  W-L=169-238  hit=41.5%  roi=-22.6%
  15%: n=359  W-L=151-208  hit=42.1%  roi=-21.8%
  20%: n=297  W-L=131-166  hit=44.1%  roi=-17.8%


In [11]:
# ── Show the large-delta cases so we can see what they actually are ─────────
print("=== Trigger signals with pts_delta >= 5 (inspect for multi-play gaps) ===")
large = triggers[triggers['pts_delta'] >= 5].sort_values('pts_delta', ascending=False)
cols = ['game_date_et','player_name','bookmaker','trigger_period','trigger_clock',
        'current_points','pts_delta','live_line','line_delta','poll_interval_s','outcome','trigger_text']
print(large[cols].to_string(index=False))


=== Trigger signals with pts_delta >= 5 (inspect for multi-play gaps) ===
game_date_et     player_name  bookmaker  trigger_period trigger_clock  current_points  pts_delta  live_line  line_delta  poll_interval_s outcome                                                            trigger_text
  2026-03-02 Bilal Coulibaly draftkings             2.0          1:13            11.0        5.0       16.5         5.0        60.343035    LOSS                             Bilal Coulibaly makes 24-foot three pointer
  2026-03-27 Bilal Coulibaly    fanduel             3.0          9:37            16.0        5.0       23.5         5.0        62.439022     WIN  Bilal Coulibaly makes 26-foot three point jumper (Leaky Black assists)
  2026-04-03    Jrue Holiday    fanduel             4.0          5:44            21.0        5.0       23.5         3.0        61.199019    LOSS     Jrue Holiday makes 27-foot three point jumper (Deni Avdija assists)
  2026-04-27     Jalen Green draftkings             2.0   

## 5. Results Overview

In [12]:
evald = triggers[triggers['outcome'].notna()].copy()

# Dedupe: one signal per (game, player, bookmaker, live_line) — keep first (earliest)
evald = evald.sort_values('ts').drop_duplicates(subset=['game_id','player_name','bookmaker','live_line'], keep='first')

def _decimal(american):
    american = float(american)
    if american >= 100: return american / 100 + 1
    return 100 / abs(american) + 1

evald['decimal_odds'] = evald['under_odds'].apply(_decimal)
BET = 100
evald['profit'] = evald.apply(lambda r: (r['decimal_odds'] - 1) * BET if r['outcome'] == 'WIN' else -BET, axis=1)

n      = len(evald)
wins   = (evald['outcome'] == 'WIN').sum()
profit = evald['profit'].sum()
roi    = profit / (n * BET) if n else 0

print('='*55)
print(f'  ALL 3PT TRIGGER UNDER SIGNALS (deduped)')
print('='*55)
print(f'  n signals:   {n}')
print(f'  W–L:         {wins}–{n-wins}')
print(f'  Hit rate:    {wins/n:.1%}' if n else '  Hit rate: n/a')
print(f'  Profit:      ${profit:+,.2f}')
print(f'  ROI:         {roi:+.1%}')
print('='*55)

  ALL 3PT TRIGGER UNDER SIGNALS (deduped)
  n signals:   407
  W–L:         169–238
  Hit rate:    41.5%
  Profit:      $-9,189.91
  ROI:         -22.6%


In [13]:
# ── By quarter ───────────────────────────────────────────────────────────────
print('\nBy quarter (trigger_period):')
q_rows = []
for period in sorted(evald['trigger_period'].dropna().unique()):
    sub = evald[evald['trigger_period'] == period]
    n_q = len(sub)
    w_q = (sub['outcome'] == 'WIN').sum()
    p_q = sub['profit'].sum()
    q_rows.append({'period': f'Q{int(period)}', 'n': n_q, 'wins': w_q,
                   'hit_rate': f"{w_q/n_q:.1%}" if n_q else 'n/a',
                   'roi': f"{p_q/(n_q*BET):+.1%}" if n_q else 'n/a'})
print(pd.DataFrame(q_rows).to_string(index=False))


By quarter (trigger_period):
period   n  wins hit_rate    roi
    Q1 107    46    43.0% -20.0%
    Q2  60    27    45.0% -15.1%
    Q3 162    70    43.2% -20.0%
    Q4  78    26    33.3% -37.2%


In [14]:
# ── Edge threshold sweep ─────────────────────────────────────────────────────
print('\nEdge threshold sweep (MIN_EDGE pp):')
sweep_rows = []
for thresh in EDGE_THRESHOLDS:
    sub = evald[evald['edge_after'] >= thresh]
    n_t = len(sub)
    if n_t == 0:
        sweep_rows.append({'min_edge': f'{thresh:.0%}', 'n': 0, 'wins': 0,
                           'hit_rate': 'n/a', 'roi': 'n/a'})
        continue
    w_t = (sub['outcome'] == 'WIN').sum()
    p_t = sub['profit'].sum()
    sweep_rows.append({
        'min_edge': f'{thresh:.0%}',
        'n': n_t,
        'wins': w_t,
        'hit_rate': f'{w_t/n_t:.1%}',
        'roi': f'{p_t/(n_t*BET):+.1%}',
    })
print(pd.DataFrame(sweep_rows).to_string(index=False))


Edge threshold sweep (MIN_EDGE pp):
min_edge   n  wins hit_rate    roi
      5% 407   169    41.5% -22.6%
     10% 407   169    41.5% -22.6%
     15% 359   151    42.1% -21.8%
     20% 297   131    44.1% -17.8%


In [15]:
# ── Edge sweep × Quarter ─────────────────────────────────────────────────────
print('\nEdge sweep × Quarter (hit rate):')
periods = sorted(evald['trigger_period'].dropna().unique())
cross_rows = []
for thresh in EDGE_THRESHOLDS:
    row = {'min_edge': f'{thresh:.0%}'}
    for p in periods:
        sub = evald[(evald['edge_after'] >= thresh) & (evald['trigger_period'] == p)]
        n_t = len(sub)
        if n_t == 0:
            row[f'Q{int(p)}'] = 'n/a'
        else:
            w_t = (sub['outcome'] == 'WIN').sum()
            row[f'Q{int(p)}'] = f'{w_t/n_t:.1%} (n={n_t})'
    cross_rows.append(row)
print(pd.DataFrame(cross_rows).to_string(index=False))


Edge sweep × Quarter (hit rate):
min_edge            Q1           Q2            Q3           Q4
      5% 43.0% (n=107) 45.0% (n=60) 43.2% (n=162) 33.3% (n=78)
     10% 43.0% (n=107) 45.0% (n=60) 43.2% (n=162) 33.3% (n=78)
     15%  44.9% (n=98) 47.9% (n=48) 41.5% (n=142) 35.2% (n=71)
     20%  52.3% (n=65) 48.9% (n=45) 42.7% (n=124) 34.9% (n=63)


In [16]:
# ── By bookmaker ─────────────────────────────────────────────────────────────
print('\nBy bookmaker:')
bk_rows = []
for bk, sub in evald.groupby('bookmaker'):
    n_b = len(sub)
    w_b = (sub['outcome'] == 'WIN').sum()
    p_b = sub['profit'].sum()
    bk_rows.append({'bookmaker': bk, 'n': n_b, 'wins': w_b,
                    'hit_rate': f'{w_b/n_b:.1%}', 'roi': f'{p_b/(n_b*BET):+.1%}'})
print(pd.DataFrame(bk_rows).sort_values('n', ascending=False).to_string(index=False))


By bookmaker:
     bookmaker   n  wins hit_rate    roi
    draftkings 191    80    41.9% -22.0%
       fanduel 169    73    43.2% -19.2%
williamhill_us  40    11    27.5% -49.2%
      fanatics   7     5    71.4% +33.4%


## 6. Proof-of-Concept Walk-Through

Concrete WIN + LOSS examples showing the full mechanism.

In [17]:
# Pull first WIN and first LOSS from the evaluated data
wins_df  = evald[evald['outcome'] == 'WIN'].sort_values(['trigger_period', 'ts'])
losses_df = evald[evald['outcome'] == 'LOSS'].sort_values(['trigger_period', 'ts'])

def print_example(label, r):
    print(f"=== {label} ===")
    print(f"  Game:          {r['game_id']} ({r['game_date_et']})")
    print(f"  Player:        {r['player_name']}")
    print(f"  Bookmaker:     {r['bookmaker']}")
    print(f"  Trigger play:  {r['trigger_text']}")
    print(f"  Quarter:       Q{int(r['trigger_period'])} | clock {r['trigger_clock']}")
    prior_line = r['live_line'] - r['line_delta']
    print(f"  Prior line:    {prior_line:.1f}")
    print(f"  Live line:     {r['live_line']:.1f}  (jumped +{r['line_delta']:.1f})")
    print(f"  Current pts:   {r['current_points']:.0f}  (+{r['pts_delta']:.0f} vs prior poll)")
    print(f"  Model edge:    {r['edge_after']:.1%}")
    print(f"  Final pts:     {r['final_pts']}")
    print(f"  Outcome:       {r['outcome']}  (UNDER {r['live_line']:.1f}, scored {r['final_pts']})")
    print()

if len(wins_df):
    print_example("WIN EXAMPLE", wins_df.iloc[0])
else:
    print("No WIN examples found.")

if len(losses_df):
    print_example("LOSS EXAMPLE", losses_df.iloc[0])
else:
    print("No LOSS examples found.")


=== WIN EXAMPLE ===
  Game:          401810703 (2026-02-26)
  Player:        Jarace Walker
  Bookmaker:     draftkings
  Trigger play:  Jarace Walker makes 25-foot three point jumper (T.J. McConnell assists)
  Quarter:       Q1 | clock 3:19
  Prior line:    15.5
  Live line:     16.5  (jumped +1.0)
  Current pts:   2  (+0 vs prior poll)
  Model edge:    32.3%
  Final pts:     16
  Outcome:       WIN  (UNDER 16.5, scored 16)

=== LOSS EXAMPLE ===
  Game:          401810703 (2026-02-26)
  Player:        Jarace Walker
  Bookmaker:     williamhill_us
  Trigger play:  Jarace Walker makes 25-foot three point jumper (T.J. McConnell assists)
  Quarter:       Q1 | clock 3:19
  Prior line:    14.5
  Live line:     15.5  (jumped +1.0)
  Current pts:   2  (+0 vs prior poll)
  Model edge:    29.9%
  Final pts:     16
  Outcome:       LOSS  (UNDER 15.5, scored 16)



In [18]:
# All trigger signals detail table
print(f"\nAll {len(evald)} evaluated trigger signals:")
for _, r in evald.sort_values(['trigger_period','game_date_et']).iterrows():
    prior_line = r['live_line'] - r['line_delta']
    print(f"  {r['game_date_et']} | {r['player_name']:<22} | Q{int(r['trigger_period'])} {r['trigger_clock']:>6} "
          f"| {r['bookmaker']:<14} | line {prior_line:.1f}→{r['live_line']:.1f} (+{r['line_delta']:.1f}) "
          f"| pts {r['current_points']:.0f} (+{r['pts_delta']:.0f}) "
          f"| edge {r['edge_after']:.1%} | final={r['final_pts']} | {r['outcome']}")



All 407 evaluated trigger signals:
  2026-02-26 | Jarace Walker          | Q1   3:19 | draftkings     | line 15.5→16.5 (+1.0) | pts 2 (+0) | edge 32.3% | final=16 | WIN
  2026-02-26 | Jarace Walker          | Q1   3:19 | williamhill_us | line 14.5→15.5 (+1.0) | pts 2 (+0) | edge 29.9% | final=16 | LOSS
  2026-03-01 | Isaac Okoro            | Q1   7:01 | draftkings     | line 9.5→10.5 (+1.0) | pts 0 (+0) | edge 13.5% | final=7 | WIN
  2026-03-01 | Gg Jackson             | Q1   9:30 | draftkings     | line 15.5→17.5 (+2.0) | pts 3 (+3) | edge 18.1% | final=11 | WIN
  2026-03-01 | Taylor Hendricks       | Q1   3:06 | williamhill_us | line 9.5→14.5 (+5.0) | pts 5 (+2) | edge 28.6% | final=19 | LOSS
  2026-03-01 | Taylor Hendricks       | Q1   3:06 | draftkings     | line 11.5→13.5 (+2.0) | pts 5 (+2) | edge 25.8% | final=19 | LOSS
  2026-03-02 | Christian Braun        | Q1   6:22 | draftkings     | line 13.5→15.5 (+2.0) | pts 5 (+3) | edge 19.5% | final=11 | WIN
  2026-03-02 | Christian B

## 7. Sample Signal Detail Table

In [19]:
cols = ['game_date_et','player_name','bookmaker','trigger_period','trigger_clock',
        'current_points','live_line','line_delta','pts_delta','edge_after',
        'final_pts','outcome']
display_df = evald[cols].copy()
display_df['trigger_period'] = display_df['trigger_period'].apply(lambda x: f'Q{int(x)}' if pd.notna(x) else '')
display_df['line_delta']     = display_df['line_delta'].apply(lambda x: f'+{x:.1f}' if pd.notna(x) else '')
display_df['pts_delta']      = display_df['pts_delta'].apply(lambda x: f'+{x:.0f}' if pd.notna(x) else '')
display_df['edge_after']     = display_df['edge_after'].apply(lambda x: f'{x:.1%}')

# ── Timestamp columns (Eastern Time) ────────────────────────────────────────
display_df['prev_poll_et'] = evald['prev_ts'].apply(
    lambda x: pd.Timestamp(x).tz_convert(ET).strftime('%H:%M:%S') if pd.notna(x) else '')
display_df['signal_poll_et'] = evald['ts'].dt.tz_convert(ET).dt.strftime('%H:%M:%S')
display_df['3pt_wall_et'] = evald['trigger_ts_utc'].apply(
    lambda x: pd.Timestamp(x).tz_convert(ET).strftime('%H:%M:%S') if x is not None else '')

# Reorder columns to put timestamps up front
ts_cols = ['prev_poll_et','3pt_wall_et','signal_poll_et']
other_cols = [c for c in display_df.columns if c not in ts_cols]
display_df = display_df[ts_cols + other_cols]

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 25)
display_df.sort_values(['game_date_et','trigger_period','player_name'])

,prev_poll_et,3pt_wall_et,signal_poll_et,game_date_et,player_name,bookmaker,trigger_period,trigger_clock,current_points,live_line,line_delta,pts_delta,edge_after,final_pts,outcome
1299,23:44:13,23:44:23,23:45:13,2026-02-25,Sam Hauser,draftkings,Q3,5:18,2.0,7.5,+3.0,+0,31.9%,5,WIN
2016,19:29:23,19:30:19,19:30:26,2026-02-26,Jarace Walker,draftkings,Q1,3:19,2.0,16.5,+1.0,+0,32.3%,16,WIN
2148,19:29:23,19:30:19,19:30:26,2026-02-26,Jarace Walker,williamhill_us,Q1,3:19,2.0,15.5,+1.0,+0,29.9%,16,LOSS
5117,21:49:57,21:50:53,21:50:55,2026-02-26,Jalen Green,williamhill_us,Q2,6:17,4.0,18.5,+1.0,+0,37.2%,9,WIN
4731,21:50:55,21:50:53,21:51:43,2026-02-26,Jalen Green,draftkings,Q2,6:17,7.0,21.5,+3.0,+3,31.6%,9,WIN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182296,22:31:11,22:31:11,22:32:09,2026-05-18,Alex Caruso,draftkings,Q4,8:53,22.0,22.5,+1.0,+3,27.5%,31,LOSS
182809,22:31:11,22:31:11,22:32:09,2026-05-18,Alex Caruso,fanduel,Q4,8:53,22.0,23.5,+2.0,+3,37.3%,31,LOSS
183404,21:38:14,21:38:18,21:39:13,2026-05-19,Dean Wade,draftkings,Q3,8:59,3.0,7.5,+3.0,+0,35.0%,10,LOSS
183467,21:47:13,21:47:33,21:48:14,2026-05-19,Sam Merrill,draftkings,Q3,5:38,6.0,12.5,+3.0,+0,23.6%,12,WIN
